In [1]:
!pip install sentence-transformers faiss-cpu openai



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os

In [5]:
documents = []

for file in os.listdir("data"):
    with open(f"data/{file}", "r", encoding="utf-8") as f:
        documents.append(f.read())

print("Loaded documents:", len(documents))

Loaded documents: 3


In [19]:

def chunk_text(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        
    return chunks
    

In [20]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(chunks)

print("Embedding shape:", embeddings.shape)

C:\Users\praty\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding shape: (38, 384)


In [21]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index ready")

FAISS index ready


In [22]:
def retrieve(query, k=5):
    query_embedding = model.encode([query])
    
    distances, indices = index.search(query_embedding, k)
    
    return [chunks[i] for i in indices[0]]

In [23]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="OPENROUTER_API_KEY"
)

In [98]:
def generate_answer(context_chunks, question):
    context = "\n\n".join(context_chunks)

    prompt = f"""
You are an assistant answering questions based only on the provided context.


Guidelines:
- Use only the information present in the context
- Do not use outside knowledge
- If the answer is not directly stated, infer simple and reasonable conclusions from the context
- Do not make unsupported assumptions
- Keep the answer concise and in bullet points
Context:
{context}

Question:
{question}

Answer:
"""
    response = client.chat.completions.create(
        model="meta-llama/llama-3-8b-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [99]:
def ask_question(question):
    retrieved = retrieve(question)

    print("\n Retrieved Context:\n")
    for i, chunk in enumerate(retrieved):
        print(f"Chunk {i+1}:\n{chunk}\n")

    answer = generate_answer(retrieved, question)

    print("\n Final Answer:\n")
    print(answer)

In [100]:
ask_question("What factors affect construction project delays?")


 Retrieved Context:

Chunk 1:
e Policy on Construction Delays (Operational Mechanisms)
Indecimal positions a system-driven approach to on-time delivery using:
- Integrated project management system
- Daily tracking of projects
- Instant flagging of deviations
- Automated task assignment
- Penalisation to reinforce accountability

Chunk 2:
er payments are made to an escrow account.
- A project manager verifies stage completion.
- Funds are disbursed to the construction partner after verification.

Purpose: reduce financial risk for customers and improve transparency and trust.

## 2) Delay Management & Accountability
### Zero-Toleranc

Chunk 3:
y construction stage.
4. Stage-Based Contractor Payments  
   - Payments released only after verified completion.
5. Transparent and Live Tracking  
   - Clear agreements and real-time online project monitoring.

## 4) Differentiators Highlighted on the Website
Indecimal highlights the following as 

Chunk 4:
 detailed design and cost plans with